## WELCOME BACK!

I guess that didn't take long. Let's get right to this: We got feedback 1 https://www.kaggle.com/competitions/feedback-prize-2021 and then we got feedback 2 https://www.kaggle.com/competitions/feedback-prize-effectiveness

I'm pretty sure we will be going back to these two competitions to pick up material. For now, let's just do an EDA of the data provided to us

![](https://static1.cbrimages.com/wordpress/wp-content/uploads/2020/02/Star-Wars-Yoda.jpg)

In [ ]:
import setuptools
import os

In [ ]:
import sys 
sys.path.append("../input/sentencetransformers/sentence-transformers-2.2.2/sentence-transformers-2.2.2") 
import sentence_transformers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import spacy
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder, util
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
import optuna
from optuna import Trial, visualization
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import pickle

In [ ]:
#%%capture
!pip uninstall fastai en-core-web-sm en-core-web-lg spacy -y -q
!pip install ../input/spacy306installwhlfiles/catalogue-2.0.3-py3-none-any.whl ../input/spacy306installwhlfiles/typer-0.3.2-py3-none-any.whl ../input/spacy306installwhlfiles/srsly-2.4.1-cp37-cp37m-manylinux2014x8664.whl ../input/spacy306installwhlfiles/smartopen-3.0.0-py3-none-any.whl ../input/spacy306installwhlfiles/pathy-0.5.2-py3-none-any.whl ../input/spacy306installwhlfiles/pydantic-1.7.3-cp37-cp37m-manylinux2014x8664.whl ../input/spacy306installwhlfiles/thinc-8.0.3-cp37-cp37m-manylinux2014x8664.whl ../input/spacy306installwhlfiles/spacy-3.0.6-cp37-cp37m-manylinux2014x8664.whl ../input/spacy306installwhlfiles/spacylegacy-3.0.5-py2.py3-none-any.whl -q
# !pip install ../input/spacy306installwhlfiles/encoreweb_lg-3.0.0-py3-none-any.whl
# !pip install ../input/spacy306installwhlfiles/encoreweb_md-3.0.0-py3-none-any.whl
!pip install ../input/spacy306installwhlfiles/en_core_web_sm-3.0.0-py3-none-any.whl -q
# !pip install ../input/spacy306installwhlfiles/spacyalignments-0.8.3-cp37-cp37m-manylinux2014x86_64.whl
# !pip install ../input/spacy306installwhlfiles/spacy_transformers-1.0.2-py2.py3-none-any.whl
# !pip install ../input/spacy306installwhlfiles/encoreweb_trf-3.0.0-py3-none-any.whl
import spacy

This time we find that we have a column of IDs, a column with the student texts - written by students who are learning English as a second language. To the right you will see 6 columns with scores in what seems to be a 1 to 5 scale at first sight.

In [ ]:
df = pd.read_csv('../input/feedback-prize-english-language-learning/train.csv')
df.head()

In [ ]:
print('The number of text_ids:', len(set(df['text_id'].values)), 
      'is equal to the number of rows in the dataframe:', df.shape[0], 
      'meaning that all text_ids are unique. We also dont have missing values\n',
     df.isnull().sum())

Let's skip the texts for now and let's instead try to understand what are the target measures we'll be identifying. Let's do them one by one starting by the distributions of the scores:

In [ ]:
for col in df.columns[2:]:
    print('checking on target measure:', col)
    print('unique values of this measure', sorted(set(df[col].values)))
print("\n All the values are scored from 1 to 5 in 0.5 increments. Let's double check the histograms as they stack next to each other\n\n")
sns.histplot(df[df.columns[2:]].melt(), x='value', hue='variable',
             multiple='dodge', shrink=.75, bins=9);

It seems like we have some nice distributions where almost everything seems to cluster near the center 3.0 value. Let's run a quick summary of the most basic statistics. Note that the standard deviation is very close. Except for vocabulary where the volatility is a bit smaller.

If you look at the sample submission you will see that all the original values are 3.0. This is a very good guess given these distribution and would be a really good guess for a naive submission!

In [ ]:
df.describe()

Another question to ask is whether there are just **bad** texts or if it's possible to score excellent in one target measure and then poorly in another one. When we look at the differences inside a row, we find that most of the texts have a difference of 1 point between its best and its worst scores. There are almost never differences in a row higher than 1.5 - only 12 texts have a difference of 2 between their best and worst scores. 

This could be a good sanity check before submission

In [ ]:
row_range = df[df.columns[2:]].max(axis = 1) - df[df.columns[2:]].min(axis = 1)
sns.histplot(row_range, shrink = 2.0)
print('rows with range higher than 1.5:', (row_range > 1.5).sum())

Let's study the characteristics 1 by 1:

## COHESION:
# From wikipedia:
Cohesion is the grammatical and lexical linking within a text or sentence that holds a text together and gives it meaning. It is related to the broader concept of coherence.

There are two main types of cohesion:

* grammatical cohesion: based on structural content

* lexical cohesion: based on lexical content and background knowledge.

Let's check what texts with bad and good cohesion look like. The first one seems to be all over the place. It goes back and forth from and to Winston Churchill, but some sentences just don't seem to go there at all. The second text seems to carry an idea better, although the grammar is not excellent.


In [ ]:
print('BAD COHESION EXAMPLE:\n')
df.sort_values(by = ["cohesion", "grammar"], ascending = [True, False], inplace = True)
print(df.reset_index().loc[0, 'full_text'])
print('\nGOOD COHESION EXAMPLE:\n')
df.sort_values(by = ["cohesion", "grammar"], ascending = [False, False], inplace = True)
print(df.reset_index().loc[0, 'full_text'])

## SYNTAX:
# From wikipedia:
In linguistics, syntax (/ˈsɪntæks/) is the study of how words and morphemes combine to form larger units such as phrases and sentences. Central concerns of syntax include word order, grammatical relations, hierarchical sentence structure (constituency), agreement, the nature of crosslinguistic variation, and the relationship between form and meaning (semantics). There are numerous approaches to syntax that differ in their central assumptions and goals.

# Analysis:

Let's choose a bad syntax example text. You will see that this is characterized by the non-existence of punctuation. We can force this into the dependency parser server and see the mess it creates. It seems like the dependency parser goes from to and all directions. Don't forget to uncomment the line in the middle to be able to see it. Scroll to the right so you can see the full sentence analysis.
DONT FORGET TO CLICK STOP TO THE LEFT TO STOP THE SERVER


In [ ]:
print('BAD SYNTAX EXAMPLE:\n')
df.sort_values(by = ["syntax", "grammar"], ascending = [True, False], inplace = True)
print(df.reset_index().loc[0, 'full_text'])
#doc = nlp(df.reset_index().loc[0, 'full_text'])
#spacy.displacy.serve(doc, style="dep")
print('\nGOOD SYNTAX EXAMPLE:\n')
df.sort_values(by = ["syntax", "grammar"], ascending = [False, False], inplace = True)
print(df.reset_index().loc[0, 'full_text'])

## VOCABULARY:
# From wikipedia:
A vocabulary is a set of familiar words within a person's language. A vocabulary, usually developed with age, serves as a useful and fundamental tool for communication and acquiring knowledge. Acquiring an extensive vocabulary is one of the largest challenges in learning a second language.

# Analysis:

It seems like this score is all about using words that exist and that are used correctly. Made up words or not using the words properly will get the student a bad score. Other than transformers I would probably just check that the words exist for this part of the competition

In [ ]:
print('BAD VOCABULARY EXAMPLE:\n')
df.sort_values(by = ["vocabulary", "grammar"], ascending = [True, False], inplace = True)
print(df.reset_index().loc[0, 'full_text'])
print('\nGOOD VOCABULARY EXAMPLE:\n')
df.sort_values(by = ["vocabulary", "grammar"], ascending = [False, False], inplace = True)
print(df.reset_index().loc[0, 'full_text'])

PHRASEOLOGY:
From wikipedia:
In linguistics, phraseology is the study of set or *fixed* expressions, such as idioms, phrasal verbs, and other types of multi-word lexical units (often collectively referred to as phrasemes), in which the component parts of the expression take on a meaning more specific than, or otherwise not predictable from, the sum of their meanings when used independently. For example, ‘Dutch auction’ is composed of the words Dutch ‘of or pertaining to the Netherlands’ and auction ‘a public sale in which goods are sold to the highest bidder’, but its meaning is not ‘a sale in the Netherlands where goods are sold to the highest bidder’; instead, the phrase has a conventionalized meaning referring to any auction where, instead of rising, the prices fall.

Analysis:
I think you would score high in this metric if you are able to use some idioms correctly. Note in this example the use of multi-word expressions such as "in contrast" or "as a result"

In [ ]:
print('\nGOOD PHRASEOLOGY EXAMPLE:\n')
df.sort_values(by = ["phraseology", "vocabulary"], ascending = [False, False], inplace = True)
print(df.reset_index().loc[0, 'full_text'])

## GRAMMAR:
# From wikipedia:
In linguistics, the grammar of a natural language is its set of structural constraints on speakers' or writers' composition of clauses, phrases, and words.

# Analysis:

Grammar is the set of constraints of the things that you can do in the language. In the second text below you will see a good example of grammar, but then the student misses in vocabulary by using expressions such as "even do" when they meant "even though".

In [ ]:
print('BAD GRAMMAR EXAMPLE:\n')
df.sort_values(by = ["grammar", "syntax"], ascending = [True, True], inplace = True)
print(df.reset_index().loc[0, 'full_text'])
print('\nGOOD GRAMMAR EXAMPLE BUT WITH BAD VOCABULARY:\n')
df.sort_values(by = ["grammar", "vocabulary"], ascending = [False, True], inplace = True)
print(df.reset_index().loc[0, 'full_text'])

And finally, let's review the last measure. Probably the most difficult to understand:
## CONVENTIONS

A linguistic convention is a principle or norm that has been adopted by a person or linguistic community about how to use, and therefore what the meaning is of, a specific term.

One convention is the word "ain't". This is a word that is acceptable in some settings, but if you go to Oxford or the hamptons... probably not.

Some conventions have to do with punctuation. Commas, hyphens.

Finally, the best example of convention would be how Master YODA speaks. **What Yoda says wrong is not**, but conventional linguistics it isn't.

Let's check the examples:

![](https://cdn.dribbble.com/users/458522/screenshots/8948281/media/03fa905d3a68d840bdcb4b5d118d7e16.jpg?compress=1&resize=800x600&vertical=top)

In [ ]:
print('BAD CONVENTIONAL LINGUISTICS:\n')
df.sort_values(by = ["conventions", "grammar"], ascending = [True, False], inplace = True)
print(df.reset_index().loc[0, 'full_text'])

## Let's do a sentence BERT analysis on the text

In [ ]:
model = SentenceTransformer("../input/sentencetransformers/multi-qa-MiniLM-L6-cos-v1/multi-qa-MiniLM-L6-cos-v1")


In [ ]:
from nltk.tokenize import sent_tokenize
text = "God is Great! I won a lottery."
print(sent_tokenize(text))
#nlp = spacy.load("../input/en-core-web-sm/en_core_web_sm/en_core_web_sm-3.2.0")

# def split_in_sentences(text):
#     doc = nlp(text)
#     return [str(sent).strip() for sent in doc.sents]

sentences_list = []
for row in tqdm(df.iterrows()):
    sentences = sent_tokenize(row[1]['full_text'])
    sentences_list.append(sentences)

In [ ]:
def embedding_finder (model, sentences_list):
    stored_embeddings = list()
    for sentences in tqdm(sentences_list):
        embeddings = model.encode(sentences, convert_to_tensor=True, show_progress_bar=False)
        embeddings = embeddings.mean(axis = 0)
        stored_embeddings.append(embeddings)
    print('ensure the length of embeddings matches the number of news articles:', len(stored_embeddings), len(sentences_list))
    return(stored_embeddings)

embeddings = embedding_finder(model, sentences_list)

In [ ]:
matrix_embeddings = pd.DataFrame([a.cpu().numpy() for a in embeddings])
matrix_embeddings.head(), matrix_embeddings.shape

In [ ]:
type(matrix_embeddings)
df[df.columns[2:]]

In [ ]:
# Optuna regression set-up
def objective(trial):
    
    train_x, test_x, train_y, test_y = train_test_split(matrix_embeddings, df[df.columns[2:]], test_size=0.22,random_state=42)
    
    param = {
        'tree_method':'gpu_hist',  # this parameter means using the GPU when training our model to speedup the training process
        'lambda': trial.suggest_loguniform('lambda', 1e-4, 2.0),
        'alpha': trial.suggest_loguniform('alpha', 1e-3, 10.0),
        'colsample_bytree': trial.suggest_categorical('colsample_bytree', [0.3,0.4,0.5,0.6,0.7,0.8,0.9, 1.0]),
        'subsample': trial.suggest_categorical('subsample', [0.4,0.5,0.6,0.7,0.8]),
        'learning_rate': trial.suggest_categorical('learning_rate', [0.008,0.01,0.012,0.014,0.016,0.018, 0.02]),
        'n_estimators': trial.suggest_int('n_estimators', 1, 300),
        'max_depth': trial.suggest_categorical('max_depth', [1,2,3,5,7,9,11,13,15,17]),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 300),
        'alpha': trial.suggest_loguniform('alpha', 1e-3, 10.0),
    }
    model = xgb.XGBRegressor(**param)
    
    model.fit(train_x,train_y,eval_set=[(test_x,test_y)],early_stopping_rounds=100,verbose=False)
    
    preds = model.predict(test_x)
    rmse = mean_squared_error(test_y, preds, squared=False)
    return rmse

In [ ]:
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=1000)
print('Number of finished trials:', len(study.trials))
print('Best trial:', study.best_trial.params)

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
#Visualize parameter importances.
optuna.visualization.plot_param_importances(study)

In [ ]:
params = study.best_params
params['tree_method'] = 'hist'

In [ ]:
kf = KFold(n_splits=4, shuffle = True)
kf.get_n_splits(matrix_embeddings)

y = df[df.columns[2:]]
xgmodels = list()
for train_index, test_index in kf.split(matrix_embeddings):
    
    reg = xgb.XGBRegressor(**params)
    
    X_train, X_test = matrix_embeddings.iloc[train_index], matrix_embeddings.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    reg.fit(X_train, y_train,eval_set=[(X_test, y_test)],early_stopping_rounds=100,verbose=True)
    xgmodels.append(reg)

In [ ]:
xgmodels

In [ ]:
pickle.dump(xgmodels, open('regressors-multi-qa-MiniLM-L6-cos-v1.pickle', 'wb'))

This is how you save a model

In [ ]:
# from sentence_transformers import SentenceTransformer
# modelPath = "local/path/to/model

# model = SentenceTransformer('bert-base-nli-stsb-mean-tokens')
# model.save(modelPath)
# model = SentenceTransformer(modelPath)